#Extraer el texto de Tik Tok

##1 — Instalar dependencias (ffmpeg + yt-dlp + whisper)

In [1]:
TIKTOK_URL = "https://vt.tiktok.com/ZSataVEaX/"
#"https://vt.tiktok.com/ZSatjpvbJ/"
OUT_DIR = "/content/tiktok"

In [2]:
!apt-get -y update

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:5 https://cli.github.com/packages stable InRelease [3,917 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [83.8 kB]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:10 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,288 kB]
Get:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [4,009 kB]
Get:14

In [3]:
!apt-get -y install ffmpeg

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 44 not upgraded.


In [4]:
!pip -q install -U yt-dlp openai-whisper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.0/182.0 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 19.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 77.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 6.8 MB/s eta 0:00:00


##2 — Descargar el TikTok (Plan A)

In [5]:
import os, glob

In [6]:
os.makedirs(OUT_DIR, exist_ok=True)

In [7]:
cmd = f'''yt-dlp -o "{OUT_DIR}/tiktok.%(ext)s" "{TIKTOK_URL}"'''
print(cmd)
!{cmd}

yt-dlp -o "/content/tiktok/tiktok.%(ext)s" "https://vt.tiktok.com/ZSataVEaX/"
[vm.tiktok] Extracting URL: https://vt.tiktok.com/ZSataVEaX/
[vm.tiktok] ZSataVEaX: Downloading webpage
[TikTok] Extracting URL: https://www.tiktok.com/@capibara_store_oficial/video/7582847487392632071?_r=1&_t=ZS-93d27UHPbTb
[TikTok] 7582847487392632071: Downloading webpage
[info] 7582847487392632071: Downloading 1 format(s): bytevc1_1080p_477288-1
[download] Destination: /content/tiktok/tiktok.mp4
[download] 100% of    5.17MiB in 00:00:02 at 1.74MiB/s


In [8]:
print("Archivos en OUT_DIR:", glob.glob(f"{OUT_DIR}/*"))

Archivos en OUT_DIR: ['/content/tiktok/tiktok.mp4']


✅ Si ves un .mp4 o .webm, seguimos.

##3 — Extraer audio (wav mono 16k)

In [9]:
import glob, os

In [10]:
videos = glob.glob(f"{OUT_DIR}/*.mp4") + glob.glob(f"{OUT_DIR}/*.webm") + glob.glob(f"{OUT_DIR}/*.mkv")
assert len(videos) > 0, "No encuentro video en OUT_DIR. Si falló la descarga, usa Plan B (subir mp4 a Drive)."

In [11]:
video_path = videos[0]
audio_path = f"{OUT_DIR}/audio.wav"

In [12]:
print("Video:", video_path)
!ffmpeg -y -i "{video_path}" -ac 1 -ar 16000 "{audio_path}"

Video: /content/tiktok/tiktok.mp4
ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enabl

In [13]:
print("Audio existe:", os.path.exists(audio_path), audio_path)

Audio existe: True /content/tiktok/audio.wav


##4 — Transcribir con Whisper

In [14]:
import whisper

In [15]:
model = whisper.load_model("small")  # prueba "base" si quieres más rápido
result = model.transcribe(audio_path, language="es")

text = result["text"].strip()
print(text[:1200])

100%|███████████████████████████████████████| 461M/461M [00:06<00:00, 74.7MiB/s]
/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


perfume que ganó el campeonato de perfumes de Chet Noir cuando hizo el mundial de perfumes, el ganador número uno por lo blue o de Parfum, qué cosa más espectacular, pero tiene que ser lo de Parfum, no el Parfum, no el de lo de Toalé, tiene que ser este que es como el que trae todo el espíritu del original, de lo de Toalé original, porque después el Parfum se fue, el aroma hizo otra cosa, pero este trae todo el espíritu de la creación original, que es una cosa bien fresca, increíblemente encantadora, diferente, super sexy, super super sexy, pero este es como concentrado y a parte tiene como un golpe extra de exclusividad, porque un perfume mucho más denso, que tiene como una parte como de algas, así como como de cuero, una cosa así, densa, rica, masculina, una cosa bien especial, este es uno de los perfumes favoritos de mi esposa de toda la vida, cada vez que me lo pongo de hecho me dice algo, qué cosa más espectacular, qué cosa más rica, siempre me dice algo, rara vez me dice, rara ve

##5 — Guardar TXT + segmentos JSON

In [19]:
import json

In [20]:
txt_path = f"{OUT_DIR}/transcripcion.txt"
with open(txt_path, "w", encoding="utf-8") as f:
    f.write(text + "\n")

segments_path = f"{OUT_DIR}/segments.json"
with open(segments_path, "w", encoding="utf-8") as f:
    json.dump(result.get("segments", []), f, ensure_ascii=False, indent=2)

In [21]:
print("OK guardado:", txt_path)
print("OK guardado:", segments_path)

OK guardado: /content/tiktok/transcripcion.txt
OK guardado: /content/tiktok/segments.json


###A) Mejorar la transcripción (rápido)
####1) Ver timestamps y segmentos (para validar)

In [22]:
import json

with open("/content/tiktok/segments.json", "r", encoding="utf-8") as f:
    segs = json.load(f)

for s in segs[:10]:
    print(f"[{s['start']:.2f}–{s['end']:.2f}] {s['text']}")

[0.00–6.12]  perfume que ganó el campeonato de perfumes de Chet Noir cuando hizo el mundial de perfumes, el
[6.12–13.52]  ganador número uno por lo blue o de Parfum, qué cosa más espectacular, pero tiene que ser lo de Parfum,
[13.52–18.84]  no el Parfum, no el de lo de Toalé, tiene que ser este que es como el que trae todo el espíritu del
[18.84–24.36]  original, de lo de Toalé original, porque después el Parfum se fue, el aroma hizo otra cosa, pero
[24.44–30.48]  este trae todo el espíritu de la creación original, que es una cosa bien fresca, increíblemente
[30.48–36.92]  encantadora, diferente, super sexy, super super sexy, pero este es como concentrado y a parte tiene como
[36.92–42.36]  un golpe extra de exclusividad, porque un perfume mucho más denso, que tiene como una parte como
[42.36–49.68]  de algas, así como como de cuero, una cosa así, densa, rica, masculina, una cosa bien especial,
[49.72–54.32]  este es uno de los perfumes favoritos de mi esposa de toda la vida, cada vez 

⛳ Esto te permite ver si “Cuando falla la Matrix.” está bien capturado o se comió contexto.

### Para guardar los substitulos
Ahora se guarda la información en el laptop

In [23]:
!whisper "/content/tiktok/audio.wav" --model small --language es --output_dir "/content/tiktok" --output_format srt
!ls -la /content/tiktok/*.srt

/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
[00:00.000 --> 00:06.120]  perfume que ganó el campeonato de perfumes de Chet Noir cuando hizo el mundial de perfumes, el
[00:06.120 --> 00:13.520]  ganador número uno Polo Blue o de Parfum qué cosa más espectacular pero tiene que ser lo de Parfum
[00:13.520 --> 00:18.840]  no el Parfum no de lo de Toalé tiene que ser este que es como el que trae todo el espíritu del
[00:18.840 --> 00:24.360]  original de lo de Toalé original porque después el Parfum se fue el aroma hizo otra cosa pero
[00:24.400 --> 00:30.480]  este trae todo el espíritu de la creación original que una cosa bien fresca increíblemente
[00:30.480 --> 00:36.920]  encantadora diferente super sexy super super sexy. Este es como concentrado y a parte tiene como
[00:36.920 --> 00:42.360]  un golpe extra de exclusividad porque un pe

se descarga la info

In [24]:
from google.colab import files
files.download("/content/tiktok/transcripcion.txt")
files.download("/content/tiktok/segments.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>